# Comparación de la semana · obstáculos y `reward_slices`

Esta semana corrimos seis experimentos, todos de 500.000 pasos con semilla 0:

| Vista | 2 obstáculos verdes (semana del 24) | 3 obstáculos de colores (semana del 25) |
|---|---|---|
| 5×5 | sin `reward_slices` · con `reward_slices` | con `reward_slices` |
| 3×3 | sin `reward_slices` · con `reward_slices` | con `reward_slices` |

![Mapas sin obstáculos, con 2 obstáculos verdes y con 3 obstáculos de colores, en dos partidas](https://raw.githubusercontent.com/FabriRandon/r2dreamer-cookie/main/notebooks/experimentos-semana-del-25/mapas_obstaculos.png)

*El mapa en dos partidas. El triángulo rojo es el agente y el círculo azul es el botón; al apretarlo aparece una galleta en una esquina de las habitaciones laterales.*

- **2 obstáculos verdes:** un cuadrado y un triángulo, los dos verdes, uno en cada habitación lateral.
- **3 obstáculos de colores:** un cuadrado verde y un triángulo morado en las habitaciones laterales, y un rombo amarillo en la habitación del botón.
- En los dos casos, los obstáculos cambian de casilla al comenzar cada partida y se quedan quietos durante ella.
- **`reward_slices=1`:** de las 16 secuencias de cada batch, una se toma alrededor de un momento en que el agente comió una galleta (si ya comió alguna).

El notebook responde dos preguntas:

1. **¿Ayuda `reward_slices`?** Compara los obstáculos verdes sin y con `reward_slices`.
2. **¿Importa cómo son los obstáculos?** Compara los verdes con los de colores, todos con `reward_slices`. Como referencia se agrega el run sin obstáculos de la semana del 15.

Al final están los videos de los seis experimentos.

**Usa CPU y ejecuta todo** (Entorno de ejecución → Ejecutar todo). Si falta algún video, la sección 5 lo crea y lo guarda en Drive; la primera vez tarda unos minutos.

## 1. Montar Drive

Usa la cuenta donde están los entrenamientos.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Leer los runs

Lee las métricas de cada run. Si un entrenamiento se retomó desde un checkpoint, se descarta lo que quedó repetido.

In [ ]:
import json
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta")

# Carpeta en Drive de cada run, por vista y condición.
RUNS = {
    ("5×5", "verdes, sin reward_slices"): "semana24/obstaculos_aleatorios5x5/distinct_s0",
    ("5×5", "verdes, con reward_slices"): "semana24/obstaculos_aleatorios5x5/distinct_sampler1_s0",
    ("5×5", "colores, con reward_slices"): "semana25/tres_obstaculos_colores5x5/distinct_sampler1_s0",
    ("5×5", "sin obstáculos, con reward_slices"): "semana15/cookie_partial_hall5_view5_s0_base_sampler1",
    ("3×3", "verdes, sin reward_slices"): "semana24/obstaculos_aleatorios3x3/distinct_s0",
    ("3×3", "verdes, con reward_slices"): "semana24/obstaculos_aleatorios3x3/distinct_sampler1_s0",
    ("3×3", "colores, con reward_slices"): "semana25/tres_obstaculos_colores3x3/distinct_sampler1_s0",
}
# Cada condición tiene siempre el mismo color, en todos los gráficos.
COLORES = {
    "verdes, sin reward_slices": "#00b894",
    "verdes, con reward_slices": "#257de0",
    "colores, con reward_slices": "#f47230",
    "sin obstáculos, con reward_slices": "#8a8a8a",
}


def leer(carpeta):
    """Evaluaciones y galletas por partida de entrenamiento, como listas de (paso, valor)."""
    evaluaciones, partidas = [], []
    with open(RAIZ / carpeta / "metrics.jsonl") as archivo:
        for linea in archivo:
            fila = json.loads(linea)
            for clave, serie in (("episode/eval_score", evaluaciones), ("episode/score", partidas)):
                if clave in fila:
                    # Al retomar se vuelve a un paso anterior: se borra lo que se repite.
                    while serie and serie[-1][0] >= fila["step"]:
                        serie.pop()
                    serie.append((fila["step"], fila[clave]))
    return evaluaciones, partidas


DATOS = {}
for (vista, condicion), carpeta in RUNS.items():
    try:
        DATOS[vista, condicion] = leer(carpeta)
        evaluaciones, partidas = DATOS[vista, condicion]
        pasos = max(evaluaciones[-1][0], partidas[-1][0])
        print(f"{vista} · {condicion}: hasta {pasos:,} pasos".replace(f"{pasos:,}", f"{pasos:,}".replace(",", ".")))
    except FileNotFoundError:
        print(f"{vista} · {condicion}: NO ENCONTRADO en {carpeta}")

In [ ]:
#@title Funciones de los gráficos
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 17, "axes.titlelocation": "left", "axes.titlepad": 14,
    "axes.labelsize": 14, "axes.labelcolor": "#555555", "axes.edgecolor": "#999999",
    "xtick.color": "#555555", "ytick.color": "#555555", "xtick.labelsize": 14, "ytick.labelsize": 14,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e3e3e3", "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 13,
})
MAXIMO = 23     # galletas por partida si el agente juega perfecto en el mapa sin obstáculos
AZAR = 0.05     # galletas por partida de un agente que elige al azar
FIN = 500_000
TEXTO = "#222222"
GRIS = "#555555"
FRANJA = "#eeede8"


def miles(x, _=None):
    return "0" if x == 0 else f"{x / 1000:.0f}k"


def con_puntos(numero):
    return f"{numero:,.0f}".replace(",", ".")


def con_coma(numero):
    return f"{numero:.1f}".replace(".", ",")


def lienzo(titulo, eje_y):
    figura, eje = plt.subplots(figsize=(13, 6.2), layout="constrained")
    eje.set_title(titulo)
    eje.set_xlabel("Pasos de entrenamiento")
    eje.set_ylabel(eje_y)
    eje.set_xlim(-12_000, 640_000)
    eje.set_xticks(range(0, FIN + 1, 100_000))
    eje.xaxis.set_major_formatter(FuncFormatter(miles))
    return figura, eje


def despegue(evaluaciones):
    """Tramo entre la última evaluación sin galletas y la primera con galletas (sin contar el paso 0)."""
    anterior = 0
    for paso, valor in evaluaciones:
        if paso > 0 and valor > 0:
            return anterior, paso
        anterior = paso
    return None


def marcar_despegues(figura, eje, lineas, vista):
    """Franja gris donde aparece la primera evaluación con galletas de cada run."""
    rotulos = []
    for condicion, _, corto in lineas:
        tramo = despegue(DATOS[vista, condicion][0]) if (vista, condicion) in DATOS else None
        if tramo is None:
            continue
        eje.axvspan(*tramo, color=FRANJA, linewidth=0, zorder=0)
        rotulos.append(eje.annotate(
            f"despega {corto}", xy=(sum(tramo) / 2, 1), xycoords=("data", "axes fraction"),
            xytext=(0, -6), textcoords="offset points", ha="center", va="top", color=GRIS))
    # Si dos rótulos se tocan, el siguiente baja una fila.
    figura.canvas.draw()
    dibujo = figura.canvas.get_renderer()
    ocupados = []
    for rotulo in sorted(rotulos, key=lambda r: r.xy[0]):
        fila = 0
        while any(rotulo.get_window_extent(dibujo).overlaps(otro) for otro in ocupados):
            fila += 1
            rotulo.xyann = (0, -6 - 20 * fila)
        ocupados.append(rotulo.get_window_extent(dibujo))


def rotular_finales(figura, eje, finales, piso=None, **estilo):
    """Escribe el valor final junto al último punto y separa los que quedarían encima.

    Ningún rótulo queda más abajo que el valor piso del eje y.
    """
    figura.canvas.draw()
    a_puntos = 72 / figura.dpi
    separacion = 1.35 * plt.rcParams["font.size"]
    minimo = -np.inf if piso is None else eje.transData.transform((0, piso))[1] * a_puntos
    alturas = sorted((eje.transData.transform((x, y))[1] * a_puntos, x, y, texto) for x, y, texto in finales)
    anterior = None
    for altura, x, y, texto in alturas:
        destino = max(altura, minimo) if anterior is None else max(altura, minimo, anterior + separacion)
        eje.annotate(texto, xy=(x, y), xytext=(12, destino - altura), textcoords="offset points",
                     va="center", color=TEXTO, **estilo)
        anterior = destino


def graficos(lineas, vista):
    """Los dos gráficos de una vista: galletas al juntar datos y evaluación del actor."""
    runs = [(condicion, nombre, corto) for condicion, nombre, corto in lineas if (vista, condicion) in DATOS]

    # 1. Galletas que comió el actor mientras juntaba los datos.
    figura, eje = lienzo("Galletas que comió el actor al juntar datos", "Galletas acumuladas (escala log)")
    eje.set_yscale("log", nonpositive="clip")
    eje.set_ylim(0.8, 14_000)
    eje.yaxis.set_major_locator(LogLocator(base=10))
    eje.yaxis.set_major_formatter(FuncFormatter(lambda y, _: con_puntos(y)))
    eje.yaxis.set_minor_formatter(NullFormatter())
    finales = []
    for condicion, nombre, _ in runs:
        partidas = DATOS[vista, condicion][1]
        pasos = np.array([0] + [paso for paso, _ in partidas])
        total = np.cumsum([0] + [valor for _, valor in partidas])
        estilo = "--" if condicion.startswith("sin obstáculos") else "-"
        eje.step(pasos, total, where="post", color=COLORES[condicion], linewidth=2.5, linestyle=estilo, label=nombre)
        eje.plot(pasos[-1], total[-1], "o", color=COLORES[condicion], markersize=8)
        finales.append((pasos[-1], max(total[-1], 0.8), con_puntos(total[-1])))
    marcar_despegues(figura, eje, runs, vista)
    rotular_finales(figura, eje, finales, fontweight="bold")
    eje.legend(loc="center left")
    plt.show()

    # 2. Evaluación del actor.
    figura, eje = lienzo("Evaluación del actor: 5 partidas cada 25k pasos", "Galletas por partida")
    eje.set_ylim(-0.8, 26.5)
    eje.set_yticks(range(0, 26, 5))
    eje.axhline(MAXIMO, color=GRIS, linestyle="--", linewidth=1.5)
    eje.annotate(f"máximo sin obstáculos ({MAXIMO})", xy=(1, MAXIMO), xycoords=("axes fraction", "data"),
                 xytext=(-4, 5), textcoords="offset points", ha="right", va="bottom", color=GRIS)
    eje.axhline(AZAR, color="#999999", linestyle="--", linewidth=1)
    eje.annotate("agente al azar (≈0,05)", xy=(0, AZAR), xycoords=("axes fraction", "data"),
                 xytext=(8, 8), textcoords="offset points", va="bottom", color=GRIS)
    finales = []
    for condicion, nombre, corto in runs:
        pasos, valores = np.array(DATOS[vista, condicion][0]).T
        estilo = "--" if condicion.startswith("sin obstáculos") else "-"
        eje.plot(pasos, valores, color=COLORES[condicion], linewidth=2.5, linestyle=estilo,
                 marker="o", markersize=7, markeredgecolor="white", markeredgewidth=1.2, label=nombre)
        finales.append((pasos[-1], valores[-1], f"{corto[0].upper() + corto[1:]}: {con_coma(valores[-1])}"))
    marcar_despegues(figura, eje, runs, vista)
    rotular_finales(figura, eje, finales, piso=1.2)  # los que terminan en 0, sobre la línea del azar
    eje.legend(loc="center left")
    plt.show()

## 3. ¿Ayuda `reward_slices`?

Los runs con **2 obstáculos verdes**, sin `reward_slices` y con `reward_slices`. Para cada vista hay dos gráficos:

1. **Galletas que comió el actor al juntar datos**, acumuladas y en escala logarítmica. Sin galletas en los datos, el modelo no tiene nada que aprender.
2. **Evaluación del actor:** cada 25.000 pasos juega cinco partidas eligiendo siempre su acción más probable.

La franja gris marca dónde despega cada run, es decir, su primera evaluación con galletas sin contar la del paso 0.

In [ ]:
PREGUNTA_1 = [
    # (condición, nombre en la leyenda, nombre corto en los rótulos)
    ("verdes, sin reward_slices", "Sin reward_slices", "sin reward_slices"),
    ("verdes, con reward_slices", "Con reward_slices", "con reward_slices"),
]

### Vista 5×5

In [ ]:
graficos(PREGUNTA_1, "5×5")

### Vista 3×3

In [ ]:
graficos(PREGUNTA_1, "3×3")

## 4. ¿Importa cómo son los obstáculos?

Todos con `reward_slices`, porque los 3 obstáculos de colores solo se corrieron así. Se comparan **2 obstáculos verdes** y **3 obstáculos de colores**. La línea gris punteada es el run **sin obstáculos** de la semana del 15, con la misma configuración. En 3×3 no hay run sin obstáculos con `reward_slices`.

Los gráficos son los mismos de la sección 3.

In [ ]:
PREGUNTA_2 = [
    ("verdes, con reward_slices", "2 obstáculos verdes", "verdes"),
    ("colores, con reward_slices", "3 obstáculos de colores", "colores"),
    ("sin obstáculos, con reward_slices", "Sin obstáculos", "sin obstáculos"),
]

### Vista 5×5

In [ ]:
graficos(PREGUNTA_2, "5×5")

### Vista 3×3

In [ ]:
graficos(PREGUNTA_2, "3×3")

## 5. Crear los videos que faltan (CPU)

Crea el video de la partida de semilla 0 para cada experimento de esta semana que no lo tenga o cuyo video sea anterior al último checkpoint. Juega el **actor aprendido** del último checkpoint, igual que en las evaluaciones.

Descarga una revisión fija del repositorio, instala las dependencias con PyTorch para CPU y usa el entorno de obstáculos de la semana de cada run. El video queda en la carpeta del run como `agente_episodio0.mp4`, el mismo nombre que usa la sección 7 de los notebooks de entrenamiento. Tarda unos minutos la primera vez y menos de un minuto por video.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

TRABAJO = Path("/content")
REPO = TRABAJO / "r2dreamer-videos"
URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"
REVISION = "bdffa746f32dac6e6d9c7f37fc8f6138a671e117"
SEMILLA_VIDEO = 0
# Cada semana entrenó con su propia versión del entorno de obstáculos.
ENTORNO = {
    "semana24": "notebooks/experimentos-semana-del-24/cookie.py",
    "semana25": "notebooks/experimentos-semana-del-25/cookie.py",
}
# Los seis experimentos de esta semana, en el orden de los videos.
VIDEOS = [
    (vista, condicion)
    for vista in ("5×5", "3×3")
    for condicion in ("verdes, sin reward_slices", "verdes, con reward_slices", "colores, con reward_slices")
]


def video_de(carpeta):
    archivo = RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4"
    try:
        return archivo if archivo.is_file() and archivo.stat().st_size > 0 else None
    except OSError:
        return None  # Drive puede estar sincronizando el archivo


def al_dia(carpeta):
    """El video está al día si se creó después del último checkpoint."""
    archivo, checkpoint = video_de(carpeta), RAIZ / carpeta / "latest.pt"
    try:
        return archivo is not None and archivo.stat().st_mtime >= checkpoint.stat().st_mtime
    except OSError:
        return False


pendientes = []
for vista, condicion in VIDEOS:
    carpeta = RUNS[vista, condicion]
    if not (RAIZ / carpeta / "latest.pt").exists():
        print(f"{vista} · {condicion}: no hay checkpoint en Drive.")
    elif not al_dia(carpeta):
        pendientes.append((vista, condicion))

if not pendientes:
    print("Los seis videos ya están hechos con el último checkpoint.")
else:
    print("Falta 1 video." if len(pendientes) == 1 else f"Faltan {len(pendientes)} videos.", "Preparando el código…")
    if not (REPO / ".git").is_dir():
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet"], cwd=REPO, check=True)
        subprocess.run(["git", "fetch", "--quiet", "--depth=1", URL, REVISION], cwd=REPO, check=True)
        subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    PYTHON = REPO / ".venv/bin/python"
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        UV = [sys.executable, "-m", "uv"]
        subprocess.run([*UV, "venv", "--quiet", "--python", "3.11", ".venv"], cwd=REPO, check=True)
        (REPO / "cpu.txt").write_text("torch==2.8.0+cpu\n")
        subprocess.run([*UV, "pip", "install", "--quiet", "--python", str(PYTHON),
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                        "--index-strategy", "unsafe-best-match", "--constraint", "cpu.txt",
                        "-e", ".[cookie]"], cwd=REPO, check=True)
    temporal = Path(tempfile.mkdtemp(prefix="videos-", dir=TRABAJO))
    for vista, condicion in pendientes:
        carpeta = RUNS[vista, condicion]
        print(f"Creando video: {vista} · {condicion}")
        shutil.copyfile(REPO / ENTORNO[carpeta.split("/")[0]], REPO / "envs/cookie.py")
        provisional = temporal / f"{carpeta.replace('/', '_')}.mp4"
        proceso = subprocess.run([str(PYTHON), "watch.py", "--logdir", str(RAIZ / carpeta),
                                  "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                 cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                 text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proceso.returncode == 0 and provisional.exists():
            shutil.copyfile(provisional, RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4")
            print("  ", proceso.stdout.strip().splitlines()[-1].split(" Video en ")[0])
        else:
            print("   No se pudo crear. Vuelve a ejecutar esta celda.")
            print("\n".join(proceso.stdout.splitlines()[-10:]))

## 6. Ver los videos

Los seis experimentos de esta semana, por vista. En cada video, a la izquierda está el mapa completo y a la derecha lo que recibe el agente. El encabezado indica las galletas y los pasos entrenados del checkpoint.

Colab no permite la pantalla completa en sus resultados: usa **Ampliar** para ver un video a todo el ancho y **Achicar** para volver.

En el 3×3 con obstáculos verdes y `reward_slices`, el último checkpoint guardado en Drive es de 370.000 pasos, aunque sus métricas llegan a 500.000. Su evaluación fue 0 durante todo el run.

In [ ]:
import html
from IPython.display import HTML, Video, display

TITULOS = {
    "verdes, sin reward_slices": ("2 obstáculos verdes", "sin reward_slices"),
    "verdes, con reward_slices": ("2 obstáculos verdes", "con reward_slices"),
    "colores, con reward_slices": ("3 obstáculos de colores", "con reward_slices"),
}
AMPLIAR = (
    "const t=this.closest('.tarjeta');"
    "const grande=t.classList.toggle('grande');"
    "this.textContent=grande?'Achicar':'Ampliar';"
    "t.scrollIntoView({block:'start',behavior:'smooth'});"
)


def tarjeta(vista, condicion):
    carpeta = RUNS[vista, condicion]
    obstaculos, slices = TITULOS[condicion]
    titulo = (f'<div class="cabeza"><div><h4>{html.escape(obstaculos)}</h4><p>{html.escape(slices)}</p></div>'
              f'<button onclick="{AMPLIAR}">Ampliar</button></div>')
    archivo = video_de(carpeta)
    if archivo is None:
        return f'<div class="tarjeta"><h4>{html.escape(obstaculos)}</h4><p>{html.escape(slices)}</p><p>No hay video. Ejecuta la sección 5.</p></div>'
    try:
        reproductor = Video(str(archivo), embed=True)._repr_html_()
    except OSError:
        return f'<div class="tarjeta"><h4>{html.escape(obstaculos)}</h4><p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p></div>'
    nota = ""
    if (vista, condicion) in DATOS:
        final = DATOS[vista, condicion][0][-1][1]
        nota = f"Evaluación final: {final:.1f} galletas por partida.".replace(".", ",", 1)
    if not al_dia(carpeta):
        nota += " ⚠️ Video anterior al último checkpoint: ejecuta la sección 5."
    return f'<div class="tarjeta">{titulo}{reproductor}<p><small>{html.escape(nota)}</small></p></div>'


estilos = (
    "<style>"
    ".vista{max-width:1200px;margin:0 0 28px}"
    ".vista h3{margin:0 0 10px}"
    ".fila{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}"
    ".tarjeta{border:1px solid #8885;border-radius:8px;padding:12px}"
    ".tarjeta.grande{grid-column:1/-1}"
    ".cabeza{display:flex;justify-content:space-between;align-items:flex-start;gap:8px}"
    ".cabeza button{font:inherit;padding:4px 12px;border:1px solid #8888;border-radius:6px;"
    "background:transparent;color:inherit;cursor:pointer}"
    ".tarjeta h4{margin:0}"
    ".tarjeta p{margin:2px 0 8px}"
    ".tarjeta video{width:100%;height:auto;image-rendering:pixelated}"
    "</style>"
)
bloques = []
for vista in ("5×5", "3×3"):
    tarjetas = "".join(tarjeta(v, condicion) for v, condicion in VIDEOS if v == vista)
    bloques.append(f'<div class="vista"><h3>Vista {vista}</h3><div class="fila">{tarjetas}</div></div>')
display(HTML(estilos + "".join(bloques)))